# NYC Taxi Fare Prediction - Phase 6: Model Evaluation
### Automatidata x New York City Taxi & Limousine Commission
---
**Goal:** Determine which of the three fitted models, Linear Regression, Random Forest, XGBoost, should be recommended to Automatidata and TLC, based on residual behavior, error concentration, and feature importance, not just aggregate RMSE/MAE/R².

**Operations (in order):**
1. Reload the feature matrix and re-run the Phase 5 split + fit pipeline
2. Residual analysis across all three models
3. Error breakdown by fare range and trip type
4. Feature importance comparison (RF vs XGBoost)
5. Final model recommendation
6. Executive summary draft

**Input:** `data/taxi_features.parquet`, 991,998 rows × 19 columns  
**Carried forward from Phase 5:** Linear Regression (RMSE \\$3.848, MAE \\$0.914, R² 0.884), Random Forest (RMSE \\$2.154, MAE \\$0.343, R² 0.964), XGBoost (RMSE \\$2.076, MAE \\$0.354, R² 0.966)  
**Output:** Model recommendation + executive summary, ready for portfolio writeup

In [ ]:
# Step 6.1: Notebook Initialization
import sys
sys.path.append("..")

from src.config import *
from sklearn.model_selection import train_test_split
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb

# Re-run Phase 5 pipeline in condensed form to recover fitted models and test-set predictions (logic already validated in Phase 5)

# Load feature matrix
df = pd.read_parquet(DATA_DIR / "taxi_features.parquet")

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

# 80/20 split (identical random_state to Phase 5)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

# Rebuild the three TransformedTargetRegressor pipelines
lr_model = TransformedTargetRegressor(
    regressor=Pipeline([
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ]),
    func=np.log1p, inverse_func=np.expm1
)

rf_model = TransformedTargetRegressor(
    regressor=RandomForestRegressor(
        n_estimators=100, max_depth=None, n_jobs=-1,
        random_state=RANDOM_STATE
    ),
    func=np.log1p, inverse_func=np.expm1
)

xgb_model = TransformedTargetRegressor(
    regressor=xgb.XGBRegressor(
        n_estimators=300, max_depth=6, learning_rate=0.1,
        n_jobs=-1, random_state=RANDOM_STATE
    ),
    func=np.log1p, inverse_func=np.expm1
)

# Fit all three
logger.info("Fitting Linear Regression...")
lr_model.fit(X_train, y_train)

logger.info("Fitting Random Forest...")
rf_model.fit(X_train, y_train)

logger.info("Fitting XGBoost...")
xgb_model.fit(X_train, y_train)

# Prediction on test set
lr_preds = lr_model.predict(X_test)
rf_preds = rf_model.predict(X_test)
xgb_preds = xgb_model.predict(X_test)

def evaluate_model(y_true: np.ndarray, y_pred:np.ndarray, name:str) -> dict:
    """
    Compute RMSE, MAE and R² for a single models predictions.

    Parameters
    ----------
    y_true: np.ndarray
        Ground-truth target values in dollar scale.
    y_pred: np.ndarray
        Predicted target values in dollar scale.
    name: str
        Model name for labeling.
        
    Returns
    -------
    dict
        Dictionary with keys ``model``, ``rmse``, ``mae``, ``r2``.
    """
    return {
        "model": name,
        "rmse": round(np.sqrt(mean_squared_error(y_true, y_pred)), 4),
        "mae": round(mean_absolute_error(y_true, y_pred), 4),
        "r2": round(r2_score(y_true, y_pred), 4)
    }

linear_results = evaluate_model(y_test, lr_preds, "Linear Regression")
rf_results = evaluate_model(y_test, rf_preds, "Random Forest")
xgb_results = evaluate_model(y_test, xgb_preds, "XGBoost")

print("\n" + "=" * 50)
print("Sanity Check Against Phase 5 Results")
print("=" * 50)
for r in [linear_results, rf_results, xgb_results]:
    print(f"  {r['model']:<20} RMSE ${r['rmse']:.3f} MAE ${r['mae']:.3f} R² {r['r2']:.4f}")
    
print("\n" + "=" * 50)
print("Test Set Shape")
print("=" * 50)
print(f"  X_test: {X_test.shape}")
print(f"  y_test: {y_test.shape}")


## Step 6.1: Notebook Initialisation ✅

All three models refit successfully. Results match Phase 5 exactly.

| Model | RMSE | MAE | R² |
|---|---|---|---|
| Linear Regression | \\$3.848 | \\$0.913 | 0.8841 |
| Random Forest | \\$2.154 | \\$0.343 | 0.9637 |
| XGBoost | \\$2.076 | \\$0.354 | 0.9663 |

Test set: 198,400 rows × 18 features. `y_test`, `lr_preds`, `rf_preds`, `xgb_preds` are all in memory and ready for residual analysis.

## Step 6.2: Residual Analysis
Examine the residuals (actual - predicted) for all three models. Aggregate metrics can hide systematic bias, a model can have a low RMSE while still consistently over- or under-predicting in specific fare ranges. Residual plots reveal that structure directly.

In [ ]:
# Step 6.2: Residual Analysis

def compute_residuals(
    y_true: np.ndarray,
    y_pred: np.ndarray
) -> pd.DataFrame:
    """
    Compute residuals and percentage error for a models predictions.

    Parameters
    ----------
    y_true: np.ndarray
        Ground-truth fare_amount values.
    y_pred: np.ndarray
        Predicted fare_amount values.
        
    Returns
    -------
    pd.DataFrame
        DataFrame with columns: actual, predicted, residual, abs_residual, pct_error.
    """
    residual = y_true - y_pred
    abs_residual = np.abs(residual)
    pct_error = (residual / y_true.replace(0, np.nan)) * 100

    return pd.DataFrame({
        "actual": y_true.values,
        "predicted": y_pred,
        "residual": residual.values,
        "abs_residual": abs_residual.values,
        "pct_error": pct_error.values
    })

def summarize_residuals(resid_df: pd.DataFrame, name: str) -> dict:
    """
    Compute summary statistics for a residual DataFrame.

    Parameters
    ----------
    resid_df: pd.DataFrame
        Output of ``compute_residuals()``.
    name: str
        Model name for labeling.

    Returns
    -------
    dict
        Summary stats: mean_residual (bias), std_residual, mean_abs_pct_error.
    """
    return {
        "model": name,
        "mean_residual": round(resid_df["residual"].mean(), 4),
        "median_residual": round(resid_df["residual"].median(), 4),
        "std_residual": round(resid_df["residual"].std(), 4),
        "mean_abs_pct_error": round(resid_df["pct_error"].abs().mean(), 2),
    }

def plot_residual_analysis(
    residuals_dict: dict[str, pd.DataFrame]
) -> None:
    """
    Plot a 3x3 grid of residual diagnostics for the three models.

    Each row is one model, with three columns:
        1. Residuals vs predicted (homoscedasticity check)
        2. Residual distribution histogram
        3. Actual vs predicted scatter plot with the y=x reference line

    Parameters
    ----------
    residuals_dict: dict[str, pd.DataFrame]
        Mapping of model name -> residual DataFrame from ``compute_residuals()``.
    """
    fig, axes = plt.subplots(3, 3, figsize=(18, 15))
    fig.suptitle("Residual Diagnostics: All Three Models", fontsize=16)

    colors = {
        "Linear Regression": PALETTE["accent"],
        "Random Forest": PALETTE["secondary"],
        "XGBoost": PALETTE["primary"]
    }

    for row, (name, resid_df) in enumerate(residuals_dict.items()):
        color = colors[name]
    
        # Clip for readability
        pred_clip = resid_df["predicted"].clip(
            upper=resid_df["predicted"].quantile(0.99)
        )
    
        resid_clip = resid_df["residual"].clip(
            lower=resid_df["residual"].quantile(0.01),
            upper=resid_df["residual"].quantile(0.99)
        )
    
        # Col 1: Residuals vs Predicted
        axes[row, 0].scatter(
            pred_clip, resid_clip,
            s=2, alpha=0.05, color=color
        )
        axes[row, 0].axhline(0, color="#374151", linestyle="--", linewidth=1)
        axes[row, 0].set_title(f"{name}: Residuals vs Predicted")
        axes[row, 0].set_xlabel("Predicted Fare ($)")
        axes[row, 0].set_ylabel("Residual ($)")
    
        # Col 2: Residual distribution
        axes[row, 1].hist(
            resid_clip, bins=80,
            color=color, edgecolor="white", linewidth=0.3
        )
        axes[row, 1].axvline(0, color="#374151", linestyle="--", linewidth=1)
        axes[row, 1].axvline(
            resid_df["residual"].mean(), color="black", linestyle="-", linewidth=1.5,
            label=f"Mean {resid_df['residual'].mean():.3f}"
        )
        axes[row, 1].set_title(f"{name}: Residual Distribution")
        axes[row, 1].set_xlabel("Residual ($)")
        axes[row, 1].set_ylabel("Frequency")
        axes[row, 1].legend(fontsize=8)
    
        # Col 3: Actual vs Predicted
        actual_clip = resid_df["actual"].clip(
            upper=resid_df["actual"].quantile(0.99)
        )
        axes[row, 2].scatter(
            actual_clip, pred_clip,
            s=2, alpha=0.05, color=color
        )
        max_val = max(actual_clip.max(), pred_clip.max())
        axes[row, 2].plot(
            [0, max_val], [0, max_val],
            color="#374151", linestyle="--", linewidth=1, label="y = x"
        )
        axes[row, 2].set_title(f"{name}: Actual vs Predicted")
        axes[row, 2].set_xlabel("Actual Fare ($)")
        axes[row, 2].set_ylabel("Predicted Fare ($)")
        axes[row, 2].legend(fontsize=8)
    
    plt.tight_layout()
    save_figure(fig, "phase6_residual_diagnostics")
    plt.show()
    
# Run
residuals_dict = {
    "Linear Regression": compute_residuals(y_test, lr_preds),
    "Random Forest": compute_residuals(y_test, rf_preds),
    "XGBoost": compute_residuals(y_test, xgb_preds)
}

residual_summary = pd.DataFrame([
    summarize_residuals(resid_df, name)
    for name, resid_df in residuals_dict.items()
])

print("\n" + "=" * 75)
print("Residual Summary")
print("=" * 75)

print(residual_summary.to_string(index=False))

plot_residual_analysis(residuals_dict)


## Step 6.2: Residual Analysis ✅

### Residual Summary

| Model | Mean Residual | Median Residual | Std Residual | Mean Abs % Error |
|---|---|---|---|---|
| Linear Regression | \\$0.066 | -\\$0.072 | \\$3.847 | 84.20\\% |
| Random Forest | \\$0.035 | \\$0.000 | \\$2.154 | 47.74\\% |
| XGBoost | \\$0.020 | \\$0.008 | \\$2.076 | 25.04\\% |

**⚠️ The MAPE numbers above are misleading and should not be quoted standalone.**
`pct_error = residual / actual`, and a large share of trips have `fare_amount` under \\$5, a \\$1 miss on a \\$3 fare is a 33\\% error but barely moves RMSE. This metric is dominated by low-fare trips and doesn't reflect real-world usability. Step 6.3 will break error down by fare bucket instead, which is the more honest way to read this.

### Plot Observations

**Bias (mean/median residual):** All three models are close to zero bias, none is systematically over- or under-predicting on average. XGBoost is the tightest (median \\$0.008), Random Forest close behind (median exactly \\$0.00, meaning the model's typical prediction is spot-on and errors are driven by a smaller set of harder cases).

**Linear Regression: Residuals vs Predicted:** Two things stand out, and one of them is a genuine finding, not an artifact.

- The flat horizontal bands at the top (\~$5.5) and bottom (\~-$8) are a **plotting artifact**, residuals were clipped to p1/p99 for readability, so extreme values pile up at the clip boundary. Ignore those bands.

- The **diagonal line falling from (\~$45, +\\$7) through (~$52, $0) down to (\~$60, -\\$8) is real, not an artifact.** This is the JFK flat-rate cluster: `actual` is fixed at \\$52 for these trips, but Linear Regression keeps predicting fare as a function of distance/duration, so as its distance-driven prediction climbs above or falls below \\$52, the residual falls in lockstep (`residual = 52 - predicted`, slope exactly -1). **This means `rate_jfk` isn't fully overriding the distance term in the linear model**. The   coefficient isn't large enough to flatten the prediction to a constant the way the real fare structure does. This is a concrete, explainable weakness of the linear model that the tree-based models don't share.

**Linear Regression: Residuals vs Predicted (general shape):** Outside the JFK cluster, there's a mild funnel, residual spread widens as predicted fare increases, i.e., LR is least reliable on longer/pricier trips, which tracks with it being a purely linear fit against `log1p_trip_distance`.

**Random Forest / XGBoost: Residuals vs Predicted:** Both are visibly tighter and more homoscedastic than Linear Regression, no fanning, no diagonal JFK artifact (both models split on `rate_jfk` directly, so they learn the \\$52 constant exactly rather than approximating it through distance). XGBoost's cloud is marginally tighter than Random Forest's, consistent with its lower RMSE.

**Residual Distributions:** All three are centered near zero. Linear Regression's distribution has a visibly heavier and wider tail (spread to ±\\$8) plus two small spikes at the clip boundaries (same JFK-driven values as above). RF and XGBoost distributions are sharply peaked and nearly symmetric, with XGBoost showing the narrowest spread of the three, visually confirming its RMSE advantage.

**Actual vs Predicted:** Linear Regression shows a visibly looser, more scattered band around `y = x`, worst at the low end (many predictions crowd above the diagonal for actual fares near \\$0–5) and at the JFK cluster (the vertical stripe at actual=\\$52 shows LR's predictions spread from \~$45–60 instead of collapsing to \\$52). Random Forest and XGBoost both hug `y = x` tightly across the full range, including at \\$52, confirming both models learned the flat-rate structure correctly.

### Key Takeaway
No model shows meaningful bias. This isn't a bias problem, it's a variance problem, and it's concentrated in specific segments (JFK/flat-rate trips for Linear Regression, and likely low-fare and long-tail trips for all three). Step 6.3 will quantify that concentration directly instead of relying on the misleading aggregate MAPE.


## Step 6.3: Error by Fare Range
Bucket the test set by `fare_amount` and compute RMSE/MAE per bucket for all three models. This directly tests the hypothesis from Step 6.2, that aggregate metrics hide where each model actually struggles by replacing the misleading global MAPE with per-bucket error that's actually interpretable.

In [ ]:
# Step 6.3: Error by Fare Range

FARE_BINS   : list[float] = [0, 5, 10, 15, 20, 30, 52, 1000]
FARE_LABELS : list[str]   = ["$0-5", "$5-10", "$10-15", "$15-20", "$20-30", "$30-52", "$52+"]

def compute_error_by_bucket(
    y_true: np.ndarray,
    preds_dict: dict[str, np.ndarray],
    bins: list[float] = FARE_BINS,
    labels: list[str] = FARE_LABELS
) -> pd.DataFrame:
    """
    Compute RMSE, MAE and trip count per fare bucket for multiple models.

    Buckets are defined on the true fare_amount so all models are compared on identical segments.
    RMSE and MAE are computed within each bucket independently, which is more informative than a
    single blended metric for models whose error is fare-dependent.

    Parameters
    ----------
    y_true: np.ndarray
        Ground-truth fare_amount values.
    preds_dict: dict[str, np.ndarray]
        Mapping of model name -> predicted values, same order as y_true
    bins: list[float]
        Bin edges for fare_amount.
    labels: list[str]
        Labels for each bin (len = len(bins) - 1).

    Returns
    -------
    pd.DataFrame
        One row per (bucket, model) with columns: fare_bucket, model, trip_count, pct_of_test_set, rmse, mae.
    """
    bucket = pd.cut(y_true, bins=bins, labels=labels, right=True)

    records = []
    for name, preds in preds_dict.items():
        df_tmp = pd.DataFrame({
            "fare_bucket": bucket,
            "actual": y_true.values,
            "predicted": preds
        })
        grouped = df_tmp.groupby("fare_bucket", observed=True)

        for label, group in grouped:
            rmse = np.sqrt(mean_squared_error(group["actual"], group["predicted"]))
            mae = mean_absolute_error(group["actual"], group["predicted"])
            records.append({
                "fare_bucket": label,
                "model": name,
                "trip_count": len(group),
                "pct_of_test_set": round(len(group) / len(y_true) * 100, 2),
                "rmse": round(rmse, 3),
                "mae": round(mae, 3)
            })

    result = pd.DataFrame(records)
    result["fare_bucket"] = pd.Categorical(
        result["fare_bucket"], categories=labels, ordered=True
    )

    return result.sort_values(["fare_bucket", "model"]).reset_index(drop=True)

def plot_error_by_bucket(error_df: pd.DataFrame) -> None:
    """
    Plot grouped bar charts of RMSE and MAE per fare bucket, one bar group per model,
    plus a bar chart of trip volume per bucket.

    Parameters
    ----------
    error_df: pd.DataFrame
        Output of ``compute_error_by_bucket()``.
    """
    models = error_df["model"].unique().tolist()
    colors = {
        "Linear Regression": PALETTE["accent"],
        "Random Forest": PALETTE["secondary"],
        "XGBoost": PALETTE["primary"]
    }

    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    fig.suptitle("Error by Fare Range", fontsize=15)

    bucket_labels = error_df["fare_bucket"].cat.categories.tolist()
    x = np.arange(len(bucket_labels))
    width = 0.25

    # Plot 1: RMSE by bucket
    for i, model in enumerate(models):
        sub = error_df[error_df["model"] == model].set_index("fare_bucket").reindex(bucket_labels)
        axes[0].bar(x + i * width, sub["rmse"], width, label=model, color=colors[model])

    axes[0].set_xticks(x + width)
    axes[0].set_xticklabels(bucket_labels, rotation=20)
    axes[0].set_title("RMSE by Fare Bucket")
    axes[0].set_ylabel("RMSE ($)")
    axes[0].legend(fontsize=8)

    # Plot 2: MAE by bucket
    for i, model in enumerate(models):
        sub = error_df[error_df["model"] == model].set_index("fare_bucket").reindex(bucket_labels)
        axes[1].bar(x + i * width, sub["mae"], width, label=model, color=colors[model])

    axes[1].set_xticks(x + width)
    axes[1].set_xticklabels(bucket_labels, rotation=20)
    axes[1].set_title("MAE by Fare Bucket")
    axes[1].set_ylabel("MAE ($)")
    axes[1].legend(fontsize=8)

    # Plot 3: Trip volume by bucket
    vol = error_df[error_df["model"] == models[0]].set_index("fare_bucket").reindex(bucket_labels)
    axes[2].bar(bucket_labels, vol["trip_count"], color=PALETTE["neutral"])
    axes[2].set_title("Test Set Volume by Fare Bucket")
    axes[2].set_ylabel("Trip Count")
    axes[2].tick_params(axis="x", rotation=20)
    axes[2].yaxis.set_major_formatter(
        plt.FuncFormatter(lambda v, _: f"{v/1_000:.0f}K")
    )

    plt.tight_layout()
    save_figure(fig, "phase6_error_by_fare_bucket")
    plt.show()

# Run
preds_dict = {
    "Linear Regression": lr_preds,
    "Random Forest"    : rf_preds,
    "XGBoost"           : xgb_preds,
}

error_by_bucket = compute_error_by_bucket(y_test, preds_dict)

print("\n" + "=" * 75)
print("Error by Fare Bucket")
print("=" * 75)

print(error_by_bucket.to_string(index=False))

print("\n" + "=" * 75)
print("RMSE Pivot (model x bucket)")
print("=" * 75)

print(error_by_bucket.pivot(index="fare_bucket", columns="model", values="rmse").to_string())

plot_error_by_bucket(error_by_bucket)


## Step 6.3: Error by Fare Range ✅

### Error by Fare Bucket

| Bucket | % of Test Set | LR RMSE | RF RMSE | XGB RMSE | LR MAE | RF MAE | XGB MAE |
|---|---|---|---|---|---|---|---|
| \\$0-5 | 13.02% | 4.508 | 2.328 | **1.174** | 0.294 | 0.169 | 0.170 |
| \\$5-10 | 43.17% | 0.789 | 0.418 | **0.388** | 0.369 | **0.202** | 0.197 |
| \\$10-15 | 20.37% | 1.179 | **0.512** | 0.763 | 0.612 | **0.303** | 0.291 |
| \\$15-20 | 8.76% | 1.595 | **0.785** | 0.805 | 0.826 | **0.425** | 0.416 |
| \\$20-30 | 7.18% | 2.363 | **1.031** | 1.072 | 1.330 | **0.566** | 0.563 |
| \\$30-52 | 6.81% | 8.221 | 1.986 | **1.882** | 4.732 | **0.654** | 0.754 |
| \\$52+ | 0.69% | 30.438 | **22.241** | 22.757 | 14.653 | **7.211** | 8.604 |

### This settles the "no clean winner" question, and the answer is not what the aggregate numbers suggested

XGBoost's overall RMSE win (2.076 vs 2.154) is not a broad, consistent edge. It comes almost entirely from one bucket: **\\$0-5**, where XGBoost's RMSE (1.174) is roughly half of Random Forest's (2.328). That bucket holds 13% of the test set but, because RMSE weights squared errors, a gap that large there dominates the global average.

Outside that bucket, the picture flips. In **\\$10-15** (20% of the test set, the second-largest bucket), Random Forest's RMSE is 0.512 against XGBoost's 0.763, XGBoost is 49% worse here. Random Forest also wins RMSE in \\$15-20, \\$20-30, and \\$52+. XGBoost only wins RMSE in \\$0-5, \\$5-10, and \\$30-52.

On MAE, the pattern is more one-sided: **Random Forest wins every bucket** except \\$0-5, where the two are effectively tied (\\$0.169 vs \\$0.170). This matches the overall Phase 5 result (RF MAE \\$0.343 vs XGBoost \\$0.354). Random Forest is the more consistently accurate model on a typical trip, XGBoost's overall RMSE edge is a product of it handling one specific segment (very short trips) much better, not being uniformly sharper.

**A caveat on the \\$52+ bucket:** RMSE here is high for all three models (22–30), but this isn't purely a model failure, the bin captures negotiated-rate and outlier trips with fares up to \\$600+ (from the Phase 2 EDA), so the target itself has huge variance in this bucket. Both tree models are handling it about as well as the data allows; Linear Regression (RMSE 30.4) is the real outlier.

### Plot Observations

**RMSE panel:** The \\$52+ bucket dwarfs everything visually, which can make it look like the decisive bucket, but it isn't, since it's only 0.69\% of the test set. The more consequential differences are the ones in \\$0-5 and \\$10-15, both mid-height bars but backed by real volume (13% and 20% of trips respectively).

**MAE panel:** Visually confirms Random Forest's green bars sit at or below XGBoost's blue bars in every bucket except \\$0-5, where they're essentially equal.

**Volume panel:** Confirms \\$5-10 and \\$10-15 together make up 63% of the test set. These two buckets, not the extremes, represent what "typical accuracy" means for this model in production. Random Forest is competitive or better than XGBoost in both.

### Implication for Step 6.6
This is a genuine trade-off, not noise: **XGBoost is the better choice if the business cares most about minimizing large errors on very short/cheap trips. Random Forest is the better choice for consistent accuracy across the bulk of realistic fares (\\$10-30), which is where most trips actually fall.** Feature importance in Step 6.5 should help explain why XGBoost specifically excels at the low end, worth checking whether it's leaning harder on a feature (likely `log1p_trip_distance` or the rate flags) that behaves more predictably for short, standard-rate trips.


## Step 6.4: Error by Trip Type
Break down error by rate code (standard vs JFK/Newark/Nassau/negotiated) and by trip duration bucket. Step 6.3 showed the RF/XGBoost trade-off is fare-range dependent. This step checks whether it's also trip-type dependent, particularly around the JFK flat-rate cluster where Linear Regression's structural weakness was identified in Step 6.2.

In [ ]:
# Step 6.4: Error by Trip Type

RATE_FLAG_COLS: list[str] = [
    "rate_jfk", "rate_newark", "rate_nassau_wc",
    "rate_negotiated", "rate_group_ride",
]

DURATION_BINS   : list[float] = [0, 5, 10, 15, 20, 30, 60, 180]
DURATION_LABELS : list[str]   = ["0-5", "5-10", "10-15", "15-20", "20-30", "30-60", "60-180"]

def derive_rate_label(X: pd.DataFrame) -> pd.Series:
    """
    Reconstruct a single categorical rate-code label from the one-hot rate_* flag columns for grouping purposes.

    Any row with all rate_* flags at 0 is standard rate (the dropped reference category from one-hot encoding in Phase 4).

    Parameters
    ----------
    X: pd.DataFrame
        Feature matrix containing the one-hot rate_* columns.

    Returns
    -------
    pd.Series
        Categorical label per row: 'Standard', 'JFK', 'Newark', 'Nassau/WC', 'Negotiated' or 'Group Ride'.
    """
    label_map = {
        "rate_jfk": "JFK",
        "rate_newark": "Newark",
        "rate_nassau_wc": "Nassau/WC",
        "rate_negotiated": "Negotiated",
        "rate_group_ride": "Group Ride",
    }

    labels = pd.Series("Standard", index=X.index)
    for col, name in label_map.items():
        labels = labels.mask(X[col] == 1, name)

    return labels

def compute_error_by_category(
    y_true: np.ndarray,
    preds_dict: dict[str, np.ndarray],
    category: pd.Series,
    cat_name: str
) -> pd.DataFrame:
    """
    Compute RMSE, MAE, and trip count per category for multiple models.

    Parameters
    ----------
    y_true: np.ndarray
        Ground-truth fare_amount values.
    preds_dict: dict[str, np.ndarray]
        Mapping of model name → predicted values, aligned to y_true.
    category: pd.Series
        Categorical grouping variable, aligned to y_true's index.
    cat_name: str
        Name of the category column for output labeling.

    Returns
    -------
    pd.DataFrame
        One row per (category, model) with trip_count, rmse, mae.
    """
    records = []
    for name, preds in preds_dict.items():
        df_tmp = pd.DataFrame({
            cat_name: category.values,
            "actual": y_true.values,
            "predicted": preds
        })
        for cat_val, group in df_tmp.groupby(cat_name, observed=True):
            rmse = np.sqrt(mean_squared_error(group["actual"], group["predicted"]))
            mae = mean_absolute_error(group["actual"], group["predicted"])
            records.append({
                cat_name: cat_val,
                "model": name,
                "trip_count": len(group),
                "rmse": round(rmse, 3),
                "mae": round(mae, 3)
            })

    return pd.DataFrame(records)

def plot_error_by_category(
    error_df: pd.DataFrame,
    cat_col: str,
    order: list[str],
    title: str
) -> None:
    """
    Plot grouped bar charts of RMSE and MAE per category, one bar group per model.

    Parameters
    ----------
    error_df: pd.DataFrame
        Output of ``compute_error_by_category()``.
    cat_col: str
        Name of the category column to plot on the x-axis.
    order: list[str]
        Category display order.
    title: str
        Figure suptitle.
    """
    models = error_df["model"].unique().tolist()
    colors = {
        "Linear Regression": PALETTE["accent"],
        "Random Forest": PALETTE["secondary"],
        "XGBoost": PALETTE["primary"],
    }

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(title, fontsize=15)

    x = np.arange(len(order))
    width = 0.25

    for i, model in enumerate(models):
        sub = (
            error_df[error_df["model"] == model]
            .set_index(cat_col)
            .reindex(order)
        )
        axes[0].bar(x + i * width, sub["rmse"], width, label=model, color=colors[model])
        axes[1].bar(x + i * width, sub["mae"], width, label=model, color=colors[model])

    axes[0].set_xticks(x + width)
    axes[0].set_xticklabels(order, rotation=20)
    axes[0].set_title("RMSE")
    axes[0].set_ylabel("RMSE ($)")
    axes[0].legend(fontsize=8)

    axes[1].set_xticks(x + width)
    axes[1].set_xticklabels(order, rotation=20)
    axes[1].set_title("MAE")
    axes[1].set_ylabel("MAE ($)")
    axes[1].legend(fontsize=8)

    plt.tight_layout()
    save_figure(fig, f"phase6_{title.lower().replace(' ', '-')}")
    plt.show()

# Run: Error by Rate Code
rate_labels = derive_rate_label(X_test)
rate_order = ["Standard", "JFK", "Newark", "Nassau/WC", "Negotiated", "Group Ride"]

error_by_rate = compute_error_by_category(y_test, preds_dict, rate_labels, "rate_type")
error_by_rate["rate_type"] = pd.Categorical(error_by_rate["rate_type"], categories=rate_order, ordered=True)
error_by_rate = error_by_rate.sort_values(["rate_type", "model"]).reset_index(drop=True)

print("\n" + "=" * 75)
print("Error by Rate Code")
print("=" * 75)
print(error_by_rate.to_string(index=False))

plot_error_by_category(error_by_rate, "rate_type", rate_order, "Error by Rate Code")

# Run: Error by Trip Duration
duration_bucket = pd.cut(
    np.expm1(X_test["log1p_trip_duration_min"]),
    bins=DURATION_BINS, labels=DURATION_LABELS, right=True
)

error_by_duration = compute_error_by_category(y_test, preds_dict, duration_bucket, "duration_bucket")
error_by_duration["duration_bucket"] = pd.Categorical(
    error_by_duration["duration_bucket"], categories=DURATION_LABELS, ordered=True
)
error_by_duration = error_by_duration.sort_values(["duration_bucket", "model"]).reset_index(drop=True)

print("\n" + "=" * 75)
print("Error by Trip Duration")
print("=" * 75)
print(error_by_duration.to_string(index=False))

plot_error_by_category(error_by_duration, "duration_bucket", DURATION_LABELS, "Error by Trip Duration")
    

## Step 6.4: Error by Trip Type ✅

### Error by Rate Code

| Rate Code | Trips | LR RMSE | RF RMSE | XGB RMSE | LR MAE | RF MAE | XGB MAE |
|---|---|---|---|---|---|---|---|
| Standard | 192,625 | 1.541 | **0.885** | 0.925 | 0.636 | **0.289** | 0.295 |
| JFK | 4,838 | 12.326 | 1.925 | **1.210** | 8.193 | **0.202** | 0.307 |
| Newark | 457 | 6.947 | 4.821 | **4.638** | 5.213 | **1.702** | 2.284 |
| Nassau/WC | 124 | 18.180 | **10.962** | 17.304 | 10.392 | **7.476** | 9.352 |
| Negotiated | 356 | 68.733 | 45.167 | **42.275** | 43.229 | **27.034** | 27.314 |
| Group Ride | 0 | - | - | - | - | - | - |

*(Group Ride had 0 trips in this test split, which is expected, only 10 trips exist in the entire dataset.)*

### Error by Trip Duration

| Duration (min) | Trips | LR RMSE | RF RMSE | XGB RMSE | LR MAE | RF MAE | XGB MAE |
|---|---|---|---|---|---|---|---|
| 0-5 | 31,860 | 5.408 | 3.828 | **3.784** | 0.560 | **0.267** | 0.291 |
| 5-10 | 58,805 | 0.976 | 0.556 | **0.490** | 0.341 | 0.199 | **0.195** |
| 10-15 | 42,777 | 1.115 | 0.781 | **0.728** | 0.488 | 0.274 | **0.262** |
| 15-20 | 25,640 | 1.456 | 0.642 | **0.600** | 0.717 | 0.344 | **0.334** |
| 20-30 | 23,926 | 2.521 | **1.361** | 1.397 | 1.330 | **0.502** | 0.502 |
| 30-60 | 14,041 | 6.040 | **2.938** | 3.024 | 3.501 | **0.807** | 0.891 |
| 60-180 | 967 | 31.411 | 14.496 | **12.605** | 15.959 | **3.016** | 4.149 |

### The JFK weakness is now quantified, and it's severe

Step 6.2's residual plot flagged that Linear Regression's diagonal-slope artifact meant it wasn't fully learning the JFK flat rate. This confirms it with numbers: LR's RMSE on JFK trips is **12.326**, over 6x Random Forest's and over 10x XGBoost's. Its MAE (\\$8.19) means the *average* JFK prediction is off by \\$8 on a fare that's fixed at \\$52. This alone should disqualify Linear Regression for any deployment scenario involving airport trips, which is 2.4% of all trips but a segment TLC almost certainly cares about specifically (it's the one rate code the business would recognize and ask about by name).

### Random Forest wins Standard, the bucket that matters most

`Standard` is 97% of the test set by rate code. Random Forest wins it on both RMSE (0.885 vs 0.925) and MAE (0.289 vs 0.295). Combined with the $10-30 fare range advantage from Step 6.3, this is the second piece of evidence that XGBoost's global RMSE win is concentrated in a minority of cases, not a general edge.

### A finding that should give you pause about XGBoost on rare rate codes

**Nassau/WC** (124 trips) is where XGBoost looks weakest relative to Random Forest: RMSE 17.304 vs RF's 10.962. XGBoost is barely better than *Linear Regression* (18.180) on this rate code. MAE tells the same story (9.352 vs 7.476). With only 124 test trips this could be noise, but it's consistent with a broader pattern across rare rate codes: XGBoost's win margin shrinks or reverses in Newark, Nassau/WC, and (on MAE) Negotiated and JFK, everywhere sample size is small. This suggests XGBoost may be overfitting to the dominant patterns in Standard/short-trip data and generalizing less reliably to underrepresented categories, while Random Forest's bagging averages out that instability. Worth keeping in mind for Step 6.6 is that "XGBoost is more accurate" story doesn't hold once you leave the majority class.

### Duration confirms a similar split to fare range, but not identically

XGBoost wins RMSE for 0-5 through 20-30 minutes, and again at 60-180. Random Forest wins RMSE at 30-60. On **MAE**, Random Forest wins or ties every single bucket, including 60-180 minutes, where its MAE (3.016) is well under XGBoost's (4.149) despite XGBoost having the better RMSE there (12.605 vs 14.496). That divergence (XGBoost better RMSE, RF better MAE, same bucket) says XGBoost is doing better on the bulk of long trips but blowing up harder on a subset of them, consistent with a model that's more sensitive to outlier/negotiated-fare trips, which overlap heavily with long-duration trips.

### Plot Observations

**Rate code panels:** Linear Regression's bars are visually disqualifying on JFK and Negotiated, it's not a close call. Between RF and XGBoost, the bars are close everywhere except Nassau/WC, where XGBoost's blue bar jumps up almost to LR's red bar, a visible, not just numerical, red flag.

**Duration panels:** The RMSE panel shows XGBoost (blue) consistently at or below Random Forest (green) through 30 minutes, then RF pulling ahead 30-60 before XGBoost regains the lead at 60-180. The MAE panel tells a cleaner story, green bars are at or below blue bars in every single bucket, most visibly at 60-180 where RF's bar is roughly 70% the height of XGBoost's.

### Running tally for Step 6.6
- **XGBoost wins:** RMSE on \\$0-30 fares, RMSE on Standard-adjacent short/medium trips, RMSE on very long trips
- **Random Forest wins:** MAE almost everywhere, RMSE on the dominant Standard rate code, RMSE on \\$10-30 fares (the bulk of typical trips), and is meaningfully more stable on rare rate codes (Nassau/WC)
- **Linear Regression:** Disqualified, the JFK and Negotiated failures alone rule it out for production, regardless of its acceptable baseline performance on Standard short trips


## Step 6.5: Feature Importance Comparison
Compare feature importances between Random Forest and XGBoost. Step 6.4's finding that XGBoost's edge concentrates in short/cheap trips while degrading on rare rate codes should have a visible signature here: if XGBoost is leaning harder on `log1p_trip_distance` and less on the rate flags, that would explain both patterns. Both models expose `.feature_importances_` on the underlying regressor (not the
`TransformedTargetRegressor` wrapper, so we reach into `.regressor_`).

In [ ]:
# Step 6.5: Feature Importance Comparison

def get_feature_importances(
    model: TransformedTargetRegressor,
    feature_names: list[str],
    model_name: str,
) -> pd.DataFrame:
    """
    Extract and normalise feature importances from a fitted TransformedTargetRegressor wrapping a tree-based model.

    Importances are pulled from ``model.regressor_.feature_importances_`` (the ``_`` suffix accesses
    the fitted inner estimator) and normalized to sum to 1 so RF and XGBoost are directly comparable regardless of
    their differing importance calculation methods (RF uses mean decrease in impurity; XGBoost defaults to gain).

    Parameters
    ----------
    model: TransformedTargetRegressor
        Fitted model wrapper.
    feature_names: list[str]
        Column names of X_train, in the order the model was fit on.
    model_name: str
        Label for the output column.

    Returns
    -------
    pd.DataFrame
        Columns: feature, importance, rank, sorted by importance descending.
    """
    raw_importances = model.regressor_.feature_importances_
    normalized = raw_importances / raw_importances.sum()

    df_imp = pd.DataFrame({
        "feature": feature_names,
        "importance": normalized
    }).sort_values("importance", ascending=False).reset_index(drop=True)

    df_imp["rank"] = df_imp.index + 1
    df_imp["model"] = model_name

    return df_imp

def plot_importance_comparison(
    rf_imp: pd.DataFrame,
    xgb_imp: pd.DataFrame,
    top_n: int = 12
) -> None:
    """
    Plot side by side horizontal bar charts of top-N feature importances for Random Forest and XGBoost,
    using a shared feature order (sorted by combined importance) so the two panels are visually comparable.

    Parameters
    ----------
    rf_imp: pd.DataFrame
        Output of ``get_feature_importances()`` for Random Forest.
    xgb_imp: pd.DataFrame
        Output of ``get_feature_importances()`` for XGBoost.
    top_n: int, optional
        Number of top features to display (default 12).
    """
    combined = (
        rf_imp.set_index("feature")["importance"]
        .add(xgb_imp.set_index("feature")["importance"], fill_value=0)
        .sort_values(ascending=False)
    )
    top_features = combined.head(top_n).index.tolist()

    rf_plot = rf_imp.set_index("feature").reindex(top_features)["importance"]
    xgb_plot = xgb_imp.set_index("feature").reindex(top_features)["importance"]

    fig, axes = plt.subplots(1, 2, figsize=(16, 7), sharey=True)
    fig.suptitle("Feature Importance: Random Forest vs XGBoost", fontsize=15)

    axes[0].barh(top_features, rf_plot, color=PALETTE["secondary"])
    axes[0].invert_yaxis()
    axes[0].set_title("Random Forest")
    axes[0].set_xlabel("Normalized Importance")

    axes[1].barh(top_features, xgb_plot, color=PALETTE["primary"])
    axes[1].invert_yaxis()
    axes[1].set_title("XGBoost")
    axes[1].set_xlabel("Normalized Importance")

    for ax, series in zip(axes, [rf_plot, xgb_plot]):
        for i, val in enumerate(series):
            ax.text(val + 0.005, i, f"{val:.3f}", va="center", fontsize=8)

    plt.tight_layout()
    save_figure(fig, "phase6_feature_importance_comparison")
    plt.show()

# Run
feature_names = X_train.columns.tolist()

rf_importance = get_feature_importances(rf_model, feature_names, "Random Forest")
xgb_importance = get_feature_importances(xgb_model, feature_names, "XGBoost")

print("\n" + "=" * 75)
print("Random Forest: Top 10 Features")
print("=" * 75)
print(rf_importance.head(10)[["rank", "feature", "importance"]].to_string(index=False))

print("\n" + "=" * 75)
print("XGBoost: Top 10 Features")
print("=" * 75)
print(xgb_importance.head(10)[["rank", "feature", "importance"]].to_string(index=False))

# Side by side comparison table
comparison = rf_importance.merge(
    xgb_importance, on="feature", suffixes = ("_rf", "_xgb")
)
comparison["importance_diff"] = comparison["importance_xgb"] - comparison["importance_rf"]
comparison = comparison.sort_values("importance_diff", ascending=False)

print("\n" + "=" * 75)
print("Biggest Importance Gaps (XGBoost minus Random Forest)")
print("=" * 75)
print(
    comparison[["feature", "importance_rf", "importance_xgb", "importance_diff"]]
    .to_string(index=False)
)

plot_importance_comparison(rf_importance, xgb_importance)


## Step 6.5: Feature Importance Comparison ✅

### Top 10 Features: Random Forest

| Rank | Feature | Importance |
|---|---|---|
| 1 | `log1p_trip_distance` | 0.7648 |
| 2 | `log1p_trip_duration_min` | 0.2140 |
| 3 | `rate_jfk` | 0.0098 |
| 4 | `rate_negotiated` | 0.0059 |
| 5 | `pickup_hour` | 0.0012 |
| 6 | `log1p_tolls_amount` | 0.0011 |
| 7 | `is_credit_card` | 0.0008 |
| 8 | `rate_newark` | 0.0006 |
| 9 | `pickup_day_of_week` | 0.0005 |
| 10 | `vendor_verifone` | 0.0003 |

### Top 10 Features: XGBoost

| Rank | Feature | Importance |
|---|---|---|
| 1 | `log1p_trip_distance` | 0.6756 |
| 2 | `log1p_trip_duration_min` | 0.2072 |
| 3 | `rate_jfk` | 0.0602 |
| 4 | `rate_negotiated` | 0.0155 |
| 5 | `rate_newark` | 0.0133 |
| 6 | `vendor_verifone` | 0.0069 |
| 7 | `is_credit_card` | 0.0067 |
| 8 | `rate_nassau_wc` | 0.0035 |
| 9 | `log1p_tolls_amount` | 0.0034 |
| 10 | `is_rush_hour` | 0.0022 |

### My hypothesis from Step 6.4 was wrong: worth stating plainly

I expected XGBoost to lean *harder* on `log1p_trip_distance` and *lighter* on the rate flags, which would've explained both its short-trip strength and its rare-rate-code weakness. The data shows the opposite on the second half: XGBoost weights **every single rate flag higher than Random Forest**, `rate_jfk` is 6x higher (0.060 vs 0.010), `rate_newark` is 20x higher (0.013 vs 0.0006), `rate_nassau_wc` is 31x higher (0.0035 vs 0.0001), `rate_negotiated` 2.6x higher. XGBoost's importance mass is more distributed across the categorical flags; Random Forest's is more concentrated in the two continuous features (0.979 combined vs XGBoost's 0.883).

### So what actually explains the Nassau/WC weakness from Step 6.4?

Not feature weighting, XGBoost *uses* `rate_nassau_wc` more than Random Forest does, yet still performs worse on that segment (RMSE 17.304 vs RF's 10.962). This points away from "XGBoost ignores rare categories" and toward a **variance/overfitting explanation instead**: with gradient boosting, each new tree fits residuals from the previous ones, so a feature that appears in only ~600 training rows (Nassau/WC is 0.06% of the full dataset) can get a split that fits noise in those few rows rather than signal, and that error compounds across boosting rounds. Random Forest's bagging averages many independently-grown trees, which tends to cancel out exactly this kind of noise-fitting on sparse categories. This is a plausible mechanism, not something directly provable from the importance numbers alone, flagging that distinction rather than overstating it.

### What the numbers do confirm

Both models agree almost exactly on the top 2: `log1p_trip_distance` and `log1p_trip_duration_min` together account for ~97-98% of predictive weight in both models. This validates the Phase 2/4 feature engineering decisions. The two engineered continuous features you'd expect to matter most. Everything below rank 2 is fine-tuning at the margins.

The **temporal features are dead weight in both models**. `pickup_hour`, `is_rush_hour`, `is_weekend`, `is_overnight`, `pickup_day_of_week` all sit at or near 0.000 importance in both RF and XGBoost. This matches the near-zero Pearson correlations flagged back in Phase 4 Step 4.8 and is worth calling out directly in the executive summary: the temporal engineering work (Step 4.2) added negligible model value once distance and duration are in the feature set, even though it looked promising in the raw EDA averages (Phase 2, Step 2.4). Distance and duration apparently already encode most of what time-of-day was proxying for.

### Plot Observations

The two bar charts are near-mirror images at the top (both dominated by the same green/blue mega-bar for `log1p_trip_distance`), but the tail tells the real story: XGBoost's right panel shows visible bars all the way down through `rate_nassau_wc`, while Random Forest's left panel is a wall of near-invisible slivers below rank 4. XGBoost distributes signal more broadly across rare categories; Random Forest concentrates almost entirely on the two continuous drivers and treats the rate flags as minor corrections.


## Step 6.6: Final Model Recommendation
Synthesize evidence from Steps 6.2–6.5 into a single recommendation. This is a judgment call, not a computed output. The goal is to make the trade-off explicit and defensible rather than pick the model with the best single number.

### Recommendation: Random Forest
This goes against the surface-level Phase 5 result, where XGBoost had the better aggregate RMSE (\\$2.076 vs \\$2.154) and R² (0.9663 vs 0.9637). Once the aggregate is decomposed, that edge doesn't hold up as a general property of the model, it's concentrated in a specific, minority slice of the data.

**Evidence, weighted by what matters for deployment:**
Evidence | Source | Favors
---------|--------|-------
Global RMSE / R² | Phase 5 | XGBoost (narrow)
Global MAE | Phase 5 | Random Forest
RMSE by fare bucket: dominant buckets (\\$10-30, 63% of trips) | Step 6.3 | Random Forest
RMSE by fare bucket: \\$0-5 only (13% of trips) | Step 6.3 | XGBoost
MAE by fare bucket: every bucket except tie at \\$0-5 | Step 6.3 | Random Forest
RMSE/MAE on Standard rate (97% of test set)	| Step 6.4 | Random Forest
RMSE/MAE on JFK (2.4% of test set) | Step 6.4 | Mixed: XGB better RMSE, RF much better MAE
Stability on rare rate codes (Nassau/WC) | Step 6.4 | Random Forest, by a wide margin
MAE by trip duration: every bucket | Step 6.4 | Random Forest
Feature importance sanity | Step 6.5 | Roughly equal: no disqualifying signal either way

The pattern across every segment-level breakdown is the same: XGBoost's global RMSE win comes from outperforming heavily on short/cheap trips (0-5 min, \\$0-5 fares), a real but narrow slice. Everywhere else, the dominant Standard rate code, the \\$10-30 fare range where most trips actually fall, every single duration bucket on MAE, and critically the rare rate codes where XGBoost showed instability in Step 6.4, Random Forest is equal or better. MAE, which is the more interpretable metric for a non-technical stakeholder ("the average prediction is off by \\$0.34" vs RMSE's harder-to-explain squared-error framing), favors Random Forest almost everywhere.

**Why this matters for TLC specifically:** if this model estimates fares before a ride begins (the stated business goal), a rider or driver sees one prediction, not an aggregate RMSE. What matters is: is the typical prediction close? Random Forest answers that better. XGBoost's strength being unusually accurate on short cheap trips is real but narrower in practical value than being reliably decent across the full range of realistic fares.

**Caveats worth stating honestly, not glossing over:**

- The margin between the two models is small in absolute terms almost everywhere except Nassau/WC and JFK MAE. This is not a landslide, and a reasonable person could pick XGBoost and defend it on the strength of the aggregate RMSE alone

- Neither model was tuned (both ran with "sensible defaults" per the Phase 5 brief). A tuned XGBoost (tighter max_depth, higher min_child_weight to reduce overfitting on sparse categories) could plausibly close or reverse the rare-rate-code gap; this recommendation is based on the models as built, not their theoretical ceiling

- Rare-rate-code sample sizes (124–457 trips) are small enough that some of the RF/XGBoost gap there could be sampling noise rather than a structural difference, flagged as a hypothesis in Step 6.5, not a proven mechanism

- Linear Regression is unambiguously disqualified due to RMSE 6-10x worse than the tree models on non-standard rate codes (Step 6.4) makes it unsuitable regardless of its interpretability advantage

**Final ranking:** Random Forest (recommended) -> XGBoost (viable alternative, better if the product specifically prioritizes short-trip accuracy) -> Linear Regression (not viable for production, retained only as an interpretable baseline).
